In [0]:
from pyspark.sql.functions import current_timestamp, col
import pandas as pd

# Read Day 2 CSV (all 350 rows)
pdf = pd.read_csv("/Volumes/nestle_dev_bronze/data_lake/landing/sql_sales_transactions_day2.csv")
df_day2 = spark.createDataFrame(pdf)

print(f"Total rows in Day 2 CSV: {df_day2.count()}")

In [0]:
# Get last watermark
result = spark.sql(
    "SELECT last_high_water_mark FROM nestle_dev_silver.control.watermark_tracking WHERE source_id = 'sql_sales_transactions'"
).collect()

if result:
    last_watermark = result[0][0]
    print(f"Last Watermark: {last_watermark}")
else:
    print("No watermark found!")

In [0]:
from pyspark.sql.functions import current_timestamp, col
import pandas as pd

# Read Day 2 CSV
pdf = pd.read_csv("/Volumes/nestle_dev_bronze/data_lake/landing/sql_sales_transactions_day2.csv")
df_day2 = spark.createDataFrame(pdf)

# Get watermark
last_watermark = spark.sql(
    "SELECT last_high_water_mark FROM nestle_dev_silver.control.watermark_tracking WHERE source_id = 'sql_sales_transactions'"
).collect()[0][0]

# FILTER: Only rows where modified_at > watermark
df_filtered = df_day2.filter(col("modified_at") > last_watermark).withColumn("transaction_id", col("transaction_id").cast("string"))

rows_to_load = df_filtered.count()
print(f"✅ Filtered rows for load: {rows_to_load}")

# Add timestamp
df_filtered = df_filtered.withColumn("ingestion_timestamp", current_timestamp())

# Write to Bronze
df_filtered.write.mode("append").saveAsTable("nestle_dev_bronze.sql_server.sales_transactions")

print(f"✅ Loaded successfully!")

In [0]:
total = spark.sql("SELECT COUNT(*) as total FROM nestle_dev_bronze.sql_server.sales_transactions").collect()[0][0]
print(f"Total rows now in Bronze: {total}")
print(f"Expected: 370 (300 from Day1 + 70 from Day2)")